# Setup

In [1]:
import pandas as pd
import requests as rq
import numpy as np

## Transaction level

Using the trans table - splitting by sector / country

> SELECT DISTINCT
xson.aid,
xson  -> '/narrative'->0->>'' as "Participating Org" ,
xson  ->> '@ref' as "ref",
xson  ->> '@role' as "role",
xson  ->> '@type' as "type"
FROM xson
JOIN trans on xson.aid = trans.aid
WHERE 
root='/iati-activities/iati-activity/participating-org' AND
trans_country = 'KG' AND 
trans_sector ='15150'
;

## Activity level

> SELECT
xson.aid,
xson  -> '/narrative'->0->>'' as "Participating Org" ,
xson  ->> '@ref' as "ref",
xson  ->> '@role' as "role",
xson  ->> '@type' as "type"
FROM xson
JOIN country on xson.aid = country.aid
WHERE country_code ='AF' and root='/iati-activities/iati-activity/participating-org'
AND xson.aid in (
SELECT DISTINCT
aid
FROM xson
WHERE
(
root='/iati-activities/iati-activity/sector' AND
xson->>'@vocabulary'='1' AND
xson->>'@code'='15123'
)
)
;

# Pull from d-portal database

In [2]:
country = '298'

In [3]:
sector = '31120'

In [4]:
url = "http://d-portal.iatistandard.org/dquery?form=csv&human=1&sql=SELECT%20DISTINCT%0Axson.aid%2C%0Axson%20%20-%3E%20%27%2Fnarrative%27-%3E0-%3E%3E%27%27%20as%20%22Participating%20Org%22%20%2C%0Axson%20%20-%3E%3E%20%27%40ref%27%20as%20%22ref%22%2C%0Axson%20%20-%3E%3E%20%27%40role%27%20as%20%22role%22%2C%0Axson%20%20-%3E%3E%20%27%40type%27%20as%20%22type%22%0AFROM%20xson%0AJOIN%20trans%20on%20xson.aid%20%3D%20trans.aid%0AWHERE%20%0Aroot%3D%27%2Fiati-activities%2Fiati-activity%2Fparticipating-org%27%20AND%0Atrans_country%20%3D%20%27" + country +"%27%20AND%20%0Atrans_sector%20%3D%27"+ sector +"%27%20AND%0Axson%20%20-%3E%3E%20%27%40role%27%20IN%20(%271%27%2C%272%27%2C%273%27%2C%274%27)%0A%3B%0A%0A"

In [5]:
data = pd.read_csv(url)

In [6]:
#prep role and org type columns
data['role'] = data['role'].fillna(-1).astype(int).astype(str).replace('-1', '')
data['type'] = data['type'].fillna(-1).astype(int).astype(str).replace('-1', '')

#get activity ID
data['iati-activity'] = data['aid'].str.split('=', n=2, expand=True)[1]

In [7]:
data

In [8]:
print("There are", data['iati-activity'].nunique(), "activities")

Check if there are any refs at all 

In [27]:
sum(data['ref'].isnull()) == len(data['ref'])

## Count no of activities for each pair

In [9]:
test = data[['Participating Org','ref','iati-activity']]

In [10]:
df = test.groupby(['Participating Org','ref'])['iati-activity'].nunique().reset_index()
df

# Use "list" to fill in missing refs/orgs

In [11]:
df = pd.read_csv("outputs/reporgs.csv")

Add most common name for each org-id

In [18]:
df

In [21]:
data

In [12]:
org_narratives =  data.merge(df, how='left',left_on='ref',right_on="org-id")[['ref','Participating Org','Most Common Narrative','Narrative Count','Source','role']].drop_duplicates()

#rename
org_narratives.rename(columns={'ref':'org-id',
        'Participating Org':'Participating Org Narrative',
        'Most Common Narrative':'List Narrative',
        'Narrative Count':'List Narrative Count',
        'Source':'List Narrative Source',
        'role':'Roles'},inplace=True)

In [11]:
org_narratives

# Output

References that are in the list

In [12]:
output = org_narratives[~org_narratives['List Narrative'].isna()] 
#group roles
output = output.groupby(['org-id','Participating Org Narrative','List Narrative','List Narrative Count','List Narrative Source'])['Roles'].apply(', '.join).reset_index()
output

In [13]:
#org_refs = org_narratives.merge(df, how='left',left_on='Participating Org',right_on="Most Common Narrative")[['ref','Participating Org','List narrative','Narrative Source','org-id','Source','role']].drop_duplicates()
#org_refs

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=710ae5c2-768a-4825-91df-2032e21c91b0' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>